# EEG channel selection

Notebook to support the paper on channel selection. 

We assume the EEG train and test sets are stored in a 3D numpy, shape ``X.shape = 
(n_cases, n_channels, n_timepoints)``, i.e. that any epoching/segmentation has already occurred
 so that each 2D array represents a single case aligned with a class label ``y.shape =
  (n_cases,)``.
  
We evaluate channel selectors on the EEG datasets in the (EEG Classification archive)[https://openreview.net/pdf?id=oPQpQlsfD0] [2]
that have more than 10 channels. These can be directly downloaded using ``aeon`` with
 the function ``load_classification``

In [17]:
import warnings

warnings.simplefilter("ignore")
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)
from aeon.datasets import load_classification
from aeon.datasets.tsc_datasets import eeg2026

print(eeg2026)

['Alzheimers', 'Blink', 'ButtonPress', 'EpilepticSeizures', 'EyesOpenShut', 'FaceDetection', 'FeedbackButton', 'FeetHands', 'FingerMovements', 'HandMovementDirection', 'ImaginedFeetHands', 'ImaginedOpenCloseFist', 'InnerSpeech', 'LongIntervalTask', 'LowCost', 'MatchingPennies', 'MindReading', 'MotorImagery', 'OpenCloseFist', 'PhotoStimulation', 'PronouncedSpeech', 'SelfRegulationSCP1', 'SelfRegulationSCP2', 'ShortIntervalTask', 'SitStand', 'Sleep', 'SongFamiliarity', 'VisualSpeech']


If already downloaded, you can specify the extract path ``X_train, y_train = 
load_classification(extract_path= "C:/Temp", split="train")``. We include some of the
 small datasets for example purposes

In [18]:
X_train, y_train = load_classification(
    extract_path="../aeon_neuro/datasets/data/", name="EyesOpenShut", split="train"
)
X_test, y_test = load_classification(
    extract_path="../aeon_neuro/datasets/data/", name="EyesOpenShut", split="test"
)
print(X_train.shape)
print(X_test.shape)

(56, 14, 128)
(42, 14, 128)


``EyesOpenShut`` has 56 train cases, each EEG has 14 channels and length 128. Channel
 selection algorithms aim to reduce the number of channels without reducing (or even 
 increasing) accuracy. We use ``aeon`` transformers to perform channel selection. Some 
 general purpose algorithms are in ``aeon``,  whereas EEG specific transforms are in 
 ``aeon-neuro``. 

In [19]:
import aeon.transformations.collection.channel_selection as cs

for name in cs.__all__:
    print(name)

ChannelScorer
ElbowClassPairwise
ElbowClassSum
RandomChannelSelector


``RandomChannelSelector`` is an unsupervised transform that simply selects a random 
proportion of channels.  ElbowClassPairwise and ElbowClassSum are supervised 
transforms proposed in [3]. 



In [25]:
from aeon.transformations.collection.channel_selection import (
    ElbowClassSum,
    RandomChannelSelector,
)

rc = RandomChannelSelector(p=1.0 / 7.0)
X_train2 = rc.fit_transform(X_train)
X_test2 = rc.transform(X_test)
print("Random selected channels ", rc.channels_selected_)
print(X_train2.shape)
print(X_test2.shape)
ecs = ElbowClassSum()
X_train2 = ecs.fit_transform(X_train, y_train)
print("ECS selected channels", ecs.channels_selected_)
X_test2 = ecs.transform(X_test)
print(X_train2.shape)
print(X_test2.shape)

Random selected channels  [3 9]
(56, 2, 128)
(42, 2, 128)
ECS selected channels [12, 13, 0]
(56, 3, 128)
(42, 3, 128)


(We cannot control the number of channels selected by ``ElbowClassSum``)
 In [3] we proposed scoring channels with a time series classifier. We can build the 
rocket scorer we used in [2] very simply using ``ChannelScorer``.


In [26]:
from aeon.classification.convolution_based import RocketClassifier
from aeon.transformations.collection.channel_selection import ChannelScorer

rocket = ChannelScorer(estimator=RocketClassifier(n_kernels=500), proportion=1.0 / 7.0)
X_train2 = rocket.fit_transform(X_train, y_train)
X_test2 = rocket.transform(X_test)
print("ROCKETScorer selected channels", rocket.channels_selected_)
print(X_train2.shape)
print(X_test2.shape)

ROCKETScorer selected channels [0 9]
(56, 2, 128)
(42, 2, 128)


We have implementations of the following channel selectors in ``aeon-neuro``.

We have some channel creation algorithms too

In [ ]:
import aeon.transformations.collection.channel_creation as cs

for name in cs.__all__:
    print(name)

In [14]:
import aeon_neuro.transformations.collection.channel_selection as cs

for name in cs.__all__:
    print(name)

C:\Code\aeon-neuro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UMAPTransformer


[1] HAIS PAPER
[2] Bhaskar Dhariyal et al. "Scalable Classifier-Agnostic Channel Selection
    for Multivariate Time Series Classification", DAMI, ECML, Springer, 2023
[3] Alejandro Pasos Ruiz and Anthony Bagnall. "Dimension selection strategies
    for multivariate time series classification with HIVE-COTEv2.0." AALTD,
    ECML-PKDD, 2022